### Notebook 8 - LLM Explanation Layer (No API Required)

#### Goal
Generate human-readable explanations for recommendations.


### Architecture

### Imports

In [8]:
import pandas as pd

### Load Files

In [16]:
items_df = pd.read_csv("data/item_df.csv")

interactions_df = pd.read_csv(
    "data/interactions.csv"
)

ranking_df = pd.read_csv(
    "data/ranking_results.csv"
)

#### Build User History Table

Merge interactions with item information.

In [17]:
user_history = interactions_df.merge(
    items_df,
    on="item_id"
)

user_history.head()

,user_id,item_id,event_type,timestamp,title,category
0,253,757,like,2026-06-12 21:46:26.155765,Content 757,Python
1,439,317,like,2025-07-09 21:46:26.155786,Content 317,Deep Learning
2,412,239,watch,2026-06-11 21:46:26.155794,Content 239,Finance
3,42,441,like,2025-12-24 21:46:26.155801,Content 441,AI
4,230,603,click,2026-05-09 21:46:26.155807,Content 603,AI


### Select User

Example:

In [18]:
user_id = 42

### User Interaction History

In [19]:
user_data = user_history[
    user_history["user_id"] == user_id
]

user_data.head()

,user_id,item_id,event_type,timestamp,title,category
3,42,441,like,2025-12-24 21:46:26.155801,Content 441,AI
119,42,371,search,2026-01-01 21:46:26.156555,Content 371,Computer Vision
606,42,427,like,2026-02-16 21:46:26.158994,Content 427,Machine Learning
869,42,497,watch,2025-12-16 21:46:26.159730,Content 497,Cloud
2274,42,630,click,2025-12-21 21:46:26.166721,Content 630,Finance


### Extract Top Interests

In [20]:
top_categories = (
    user_data["category"]
    .value_counts()
    .head(3)
    .index
    .tolist()
)

print(top_categories)

['Machine Learning', 'LLM', 'Cloud']


### Get Ranked Recommendations

From Notebook 6:

In [21]:
top_recommendations = ranking_df.merge(
    items_df,
    on="item_id"
)

top_recommendations = top_recommendations.sort_values(
    "rank_score",
    ascending = False
)

### Create Industry-Level Explanation Function

In [25]:
def generate_explanation(
    user_categories,
    item_category,
    rank_score
):

    if item_category in user_categories:

        return (
            f"Recommended because you frequently engage with "
            f"{item_category}-related content."
        )

    if rank_score > 0:

        return (
            "Recommended based on your recent interaction patterns "
            "and similarity to previously consumed content."
        )

    return (
        "Recommended because similar users found this content relevant."
    )

### Generate Explanations

In [26]:
top_recommendations["explanation"] = top_recommendations.apply(
    lambda row:
    generate_explanation(
        top_categories,
        row["category"],
        row["rank_score"]
    ),
    axis=1
)

### View Results

In [27]:
top_recommendations[
    [
        "title",
        "category",
        "rank_score",
        "explanation"
    ]
].head(10)

,title,category,rank_score,explanation
3044,Content 768,Deep Learning,1.249428,Recommended based on your recent interaction p...
7249,Content 768,Deep Learning,1.246384,Recommended based on your recent interaction p...
8688,Content 468,Business,1.233394,Recommended based on your recent interaction p...
5152,Content 276,Business,1.212895,Recommended based on your recent interaction p...
1004,Content 658,LLM,1.195624,Recommended because you frequently engage with...
8619,Content 633,Machine Learning,1.186485,Recommended because you frequently engage with...
4540,Content 132,Cloud,1.134260,Recommended because you frequently engage with...
4142,Content 768,Deep Learning,1.087603,Recommended based on your recent interaction p...
550,Content 786,Computer Vision,1.080158,Recommended based on your recent interaction p...
1240,Content 659,Business,1.075504,Recommended based on your recent interaction p...


### Pretty Output

In [28]:
for _, row in top_recommendations.head(10).iterrows():

    print("=" * 80)

    print("Title:", row["title"])
    print("Category:", row["category"])
    print("Rank Score:", round(row["rank_score"], 4))
    print("Reason:", row["explanation"])

Title: Content 768
Category: Deep Learning
Rank Score: 1.2494
Reason: Recommended based on your recent interaction patterns and similarity to previously consumed content.
Title: Content 768
Category: Deep Learning
Rank Score: 1.2464
Reason: Recommended based on your recent interaction patterns and similarity to previously consumed content.
Title: Content 468
Category: Business
Rank Score: 1.2334
Reason: Recommended based on your recent interaction patterns and similarity to previously consumed content.
Title: Content 276
Category: Business
Rank Score: 1.2129
Reason: Recommended based on your recent interaction patterns and similarity to previously consumed content.
Title: Content 658
Category: LLM
Rank Score: 1.1956
Reason: Recommended because you frequently engage with LLM-related content.
Title: Content 633
Category: Machine Learning
Rank Score: 1.1865
Reason: Recommended because you frequently engage with Machine Learning-related content.
Title: Content 132
Category: Cloud
Rank Scor

### Save Final Recommendations

In [29]:
top_recommendations.to_csv(
    "data/final_recommendations.csv",
    index=False
)

### Final Pipeline